In [1]:
# imports
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# LOAD DELIVERABLES (ALREADY LAGGED)
df = pd.read_parquet("data/fx_system_df_lagged.parquet")
returns_df = pd.read_parquet("data/fx_returns_df_lagged.parquet")

# CONFIG (FROZEN SPEC)
H = 21
MIN_CROSS_SECTION = 5
ROLL_WINDOW = 252
LOOKBACK = 126

DISP_SMOOTH = 21
DISP_Z_WINDOW = 252

# FX MAP
FX_TICKERS = {
    "AUD": {"ticker": "AUDUSD=X", "invert": False},
    "CAD": {"ticker": "CADUSD=X", "invert": True}, 
    "CHF": {"ticker": "CHFUSD=X", "invert": True},
    "EUR": {"ticker": "EURUSD=X", "invert": False},
    "GBP": {"ticker": "GBPUSD=X", "invert": False},
    "JPY": {"ticker": "JPYUSD=X", "invert": True}, 
    "NZD": {"ticker": "NZDUSD=X", "invert": False}
}

# BUILD TREND_CS
def build_trend_cs(df, returns_df, fx_map):
    trend = pd.DataFrame(index=df.index)

    # --- RAW MOMENTUM ---
    for ccy in fx_map.keys():
        price = df[f"{ccy}_price"]
        trend[ccy] = np.log(price).diff(LOOKBACK)

    # --- ALIGN ---
    common_idx = trend.index.intersection(returns_df.index)
    trend = trend.loc[common_idx]
    ret = returns_df.loc[common_idx]

    # --- VOL ADJUST (TIME-SERIES NORMALIZATION) ---
    vol = ret.rolling(LOOKBACK).std().replace(0, np.nan)
    trend = trend / vol

    # --- CROSS-SECTIONAL Z-SCORE (REMOVE USD + STANDARDIZE) ---
    cs_mean = trend.mean(axis=1)
    cs_std = trend.std(axis=1).replace(0, np.nan)

    trend = trend.sub(cs_mean, axis=0).div(cs_std, axis=0)

    # --- DISPERSION GATE (t−1, PURE SCALING ONLY) ---
    disp = ret.std(axis=1)
    disp = disp.rolling(DISP_SMOOTH).mean()

    z = (disp - disp.rolling(DISP_Z_WINDOW).mean()) / disp.rolling(DISP_Z_WINDOW).std()
    gate = z.shift(1).clip(lower=0, upper=1)

    trend = trend.mul(gate, axis=0)

    return trend


# FORWARD RETURNS
def compute_forward_returns(returns_df, H):
    return returns_df.rolling(H).sum().shift(-H)


# IC SERIES
def compute_ic_series(signal, future_returns):
    ic_vals = []

    for date in signal.index:
        s = signal.loc[date]
        r = future_returns.loc[date]

        valid = s.notna() & r.notna()

        if valid.sum() < MIN_CROSS_SECTION:
            ic_vals.append(np.nan)
            continue

        if s[valid].std() == 0:
            ic_vals.append(np.nan)
            continue

        ic_vals.append(spearmanr(s[valid], r[valid]).statistic)

    return pd.Series(ic_vals, index=signal.index)


# BUILD SIGNAL
trend_cs_signal = build_trend_cs(df, returns_df, FX_TICKERS)

# ALIGN
common_idx = trend_cs_signal.index.intersection(returns_df.index)
trend_cs_signal = trend_cs_signal.loc[common_idx].copy()
returns_df = returns_df.loc[common_idx].copy()

# DROP LOW COVERAGE
valid_counts = trend_cs_signal.notna().sum(axis=1)
trend_cs_signal = trend_cs_signal[valid_counts >= MIN_CROSS_SECTION]
returns_df = returns_df.loc[trend_cs_signal.index]

# DAILY SIGNAL FOR PORTFOLIO CONSTRUCTION
trend_cs_signal_daily = trend_cs_signal.copy()

# FORWARD RETURNS
future_returns = compute_forward_returns(returns_df, H)

# NON-OVERLAPPING IC DIAGNOSTIC ONLY
future_returns_ic = future_returns.iloc[::H]

# ALIGN IC INPUTS
common_idx = trend_cs_signal.index.intersection(future_returns_ic.index)
trend_cs_signal_ic = trend_cs_signal.loc[common_idx]
future_returns_ic = future_returns_ic.loc[common_idx]

# IC SERIES
ic_series = compute_ic_series(trend_cs_signal_ic, future_returns_ic)

# CLEAN
ic_series_clean = ic_series.dropna()

# EXPANDING IC
expanding_ic_mean = ic_series_clean.expanding().mean()
expanding_ic_std = ic_series_clean.expanding().std()
expanding_ic_ir = expanding_ic_mean / expanding_ic_std

# ROLLING IC
rolling_ic_mean = ic_series_clean.rolling(ROLL_WINDOW).mean()
rolling_ic_std = ic_series_clean.rolling(ROLL_WINDOW).std()
rolling_ic_ir = rolling_ic_mean / rolling_ic_std

# IC DISTRIBUTION
ic_stats = {
    "IC Mean": ic_series_clean.mean(),
    "IC Median": ic_series_clean.median(),
    "IC > 0 %": (ic_series_clean > 0).mean(),
}

# EFFECTIVE SAMPLE SIZE
n_eff = len(ic_series_clean)

# FINAL OUTPUT
results = {
    "IC Mean": ic_stats["IC Mean"],
    "IC Median": ic_stats["IC Median"],
    "IC > 0 %": ic_stats["IC > 0 %"],
}

results_df = pd.DataFrame([results])

# SAVE
trend_cs_signal_daily.to_parquet("data/trend_cs_signal.parquet")
ic_series.to_frame("IC").to_parquet("data/trend_cs_ic_series.parquet")
results_df.to_parquet("data/trend_cs_summary.parquet")

In [2]:
results_df

,IC Mean,IC Median,IC > 0 %
0,0.096241,0.214286,0.565789
